In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def smape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(2*np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)))

def eval_metrics(name, y_true, y_pred):
    return {
        "model": name,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(root_mean_squared_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
        "SMAPE": float(smape(y_true, y_pred)),
    }

def split_by_day(df, test_days=7, val_days=7):
    max_day = int(df["d"].max())
    test_start = max_day - test_days + 1
    val_start  = test_start - val_days

    train_df = df[df["d"] < val_start].copy()
    val_df   = df[(df["d"] >= val_start) & (df["d"] < test_start)].copy()
    test_df  = df[df["d"] >= test_start].copy()

    return train_df, val_df, test_df

# 補零
def densify_topk_series(df_raw: pd.DataFrame, top_k=20000):
    """
    df_raw 需要欄位: d,t,x,y,count
    回傳：只含 top_k 個 (x,y,t) 且已補齊所有 d 的 DataFrame
    """
    df = df_raw[["d","t","x","y","count"]].copy()

    # 選最活躍的 (x,y,t)：用總量或出現天數都可以
    key_sum = df.groupby(["x","y","t"])["count"].sum().sort_values(ascending=False)
    top_keys = key_sum.head(top_k).index

    df = df.set_index(["x","y","t"]).loc[top_keys].reset_index()

    dmin, dmax = int(df["d"].min()), int(df["d"].max())
    all_d = np.arange(dmin, dmax + 1, dtype=int)

    out = []
    for (x,y,t), g in df.groupby(["x","y","t"], sort=False):
        g2 = g.set_index("d").reindex(all_d)
        g2["count"] = g2["count"].fillna(0.0)
        g2["d"] = all_d
        g2["x"] = x; g2["y"] = y; g2["t"] = t
        out.append(g2[["d","t","x","y","count"]])

    return pd.concat(out, ignore_index=True)

In [ ]:
df = pd.read_parquet("../data/processed/sapporo_density.parquet")
df = df[~((df["x"]==999) & (df["y"]==999))].copy()

df_raw = df.copy()

# 只補 top_k，先用小一點，穩了再加
df_dense = densify_topk_series(df_raw, top_k=5000)
print(df_dense.shape, df_dense["count"].mean(), (df_dense["count"]==0).mean())

(375000, 5) 11.260272 0.0011893333333333333


In [ ]:
# LSTM 序列長度
SEQ_LEN = 28

df = df_dense.copy() #如果要看沒補零的效能這行可以註解起來

# 假設 d=0 是 2023-01-01）
df["date"] = pd.to_datetime("2023-01-01") + pd.to_timedelta(df["d"], unit="D")
df["weekday"] = df["date"].dt.weekday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

# 排序
df = df.sort_values(["x","y","t","d"])

# 產生 lag_1..lag_SEQ_LEN（給 LSTM 當序列，也給 baseline）
g = df.groupby(["x","y","t"])["count"]
for k in range(1, SEQ_LEN+1):
    df[f"lag_{k}"] = g.shift(k)

# rolling 特徵（給 LGBM 用）
df["rolling_3"] = g.transform(lambda s: s.shift(1).rolling(3).mean())
df["rolling_7"] = g.transform(lambda s: s.shift(1).rolling(7).mean())

# 丟掉 lag 不足的列
need_cols = [f"lag_{k}" for k in range(1, SEQ_LEN+1)] + ["rolling_3","rolling_7"]
df = df.dropna(subset=need_cols).copy()

train_df, val_df, test_df = split_by_day(df, test_days=7, val_days=7)
print(len(train_df), len(val_df), len(test_df))

165000 35000 35000


In [4]:
def baseline_lagk(df_split, k=7):
    return df_split[f"lag_{k}"].to_numpy()

def fit_baseline_hist(train_df):
    # (weekday,t,x,y) 的歷史平均
    hist = train_df.groupby(["weekday","t","x","y"])["count"].mean()
    return hist

def predict_baseline_hist(df_split, hist_series):
    key = list(zip(df_split["weekday"], df_split["t"], df_split["x"], df_split["y"]))
    # 沒看過的 key 用全域平均補
    global_mean = float(hist_series.mean())
    pred = np.array([hist_series.get(k, global_mean) for k in key], dtype=float)
    return pred

In [5]:
import lightgbm as lgb

LGB_FEATURES = ["weekday","t","x","y","is_weekend","lag_1","lag_7","rolling_3","rolling_7"]

def train_lgbm(train_df, val_df, sample_n=800_000, seed=42):
    tr = train_df.sample(n=min(sample_n, len(train_df)), random_state=seed)
    X_tr, y_tr = tr[LGB_FEATURES], tr["count"]
    X_va, y_va = val_df[LGB_FEATURES], val_df["count"]

    model = lgb.LGBMRegressor(
        n_estimators=4000,
        learning_rate=0.05,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=4
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(50)]
    )
    return model

def predict_lgbm(model, df_split):
    return model.predict(df_split[LGB_FEATURES])

In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class SeqDatasetEmbed(Dataset):
    def __init__(self, df, seq_len):
        self.seq_len = seq_len
        # 序列：lag_seq_len ... lag_1（越後面越接近現在）
        lag_cols = [f"lag_{k}" for k in range(seq_len, 0, -1)]
        self.X_seq = df[lag_cols].to_numpy(dtype=np.float32)[:, :, None]  # (N,L,1)

        # ids
        self.weekday = df['weekday'].to_numpy(dtype=np.int64)
        self.t_id = df['t'].to_numpy(dtype=np.int64)
        self.x_id = df['x'].to_numpy(dtype=np.int64)
        self.y_id = df['y'].to_numpy(dtype=np.int64)
        self.is_weekend = df["is_weekend"].to_numpy(dtype=np.float32)

        # 目標：log1p(count) 穩定訓練
        self.y = np.log1p(df["count"].to_numpy(dtype=np.float32))

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return (
            torch.from_numpy(self.X_seq[i]),
            torch.tensor(self.weekday[i], dtype=torch.long),
            torch.tensor(self.t_id[i], dtype=torch.long),
            torch.tensor(self.x_id[i], dtype=torch.long),
            torch.tensor(self.y_id[i], dtype=torch.long),
            torch.tensor([self.is_weekend[i]], dtype=torch.float32),  # shape (1,)
            torch.tensor(self.y[i], dtype=torch.float32),
        )

class LSTMRegEmbed(nn.Module):
    def __init__(self, hidden=64, layers=1, n_weekday=7, n_t=48, n_x=2000, n_y=2000,
                 emb_wd=2, emb_t=8, emb_x=16, emb_y=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, num_layers=layers,
                            batch_first=True, dropout=0.1 if layers > 1 else 0.0)
        
        # embeddings
        self.emb_weekday = nn.Embedding(n_weekday, emb_wd)
        self.emb_t = nn.Embedding(n_t, emb_t)
        self.emb_x = nn.Embedding(n_x, emb_x)
        self.emb_y = nn.Embedding(n_y, emb_y)

        static_dim = emb_wd + emb_t + emb_x + emb_y + 1  # + is_weekend(1)

        self.head = nn.Sequential(
            nn.Linear(hidden + static_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )

    def forward(self, x_seq, weekday, t_id, x_id, y_id, is_weekend):
        out, _ = self.lstm(x_seq)          # (B,L,H)
        h = out[:, -1, :]                  # last hidden
        e = torch.cat([
            self.emb_weekday(weekday),
            self.emb_t(t_id),
            self.emb_x(x_id),
            self.emb_y(y_id),
            is_weekend,  # already (B,1)
        ], dim=1)

        z = torch.cat([h, e], dim=1)
        return self.head(z).squeeze(1)     # log1p(count)

In [12]:
def train_lstm_embed(train_df, val_df, seq_len, sample_n=800_000, batch_size=1024, epochs=50, lr=1e-3, seed=42):
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tr = train_df.sample(n=min(sample_n, len(train_df)), random_state=seed)
    ds_tr = SeqDatasetEmbed(tr, seq_len)
    ds_va = SeqDatasetEmbed(val_df, seq_len)
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=0)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=0)

    n_x = int(pd.concat([train_df["x"], val_df["x"]]).max()) + 1
    n_y = int(pd.concat([train_df["y"], val_df["y"]]).max()) + 1
    n_t = int(pd.concat([train_df["t"], val_df["t"]]).max()) + 1

    model = LSTMRegEmbed(hidden=64, layers=1, n_x=n_x, n_y=n_y, n_t=n_t).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    best_rmse = float("inf")
    best_state = None
    patience, bad = 3, 0

    for ep in range(1, epochs+1):
        model.train()
        for xseq, wd, tid, xid, yid, isw, y in dl_tr:
            xseq, wd, tid, xid, yid, isw, y = (
                xseq.to(device), wd.to(device), tid.to(device), xid.to(device),
                yid.to(device), isw.to(device), y.to(device)
            )
            pred = model(xseq, wd, tid, xid, yid, isw)
            loss = loss_fn(pred, y)
            opt.zero_grad()
            loss.backward()
            opt.step()

        # val RMSE（在原始 count 尺度）
        model.eval()
        preds, ys = [], []
        with torch.no_grad():
            for xseq, wd, tid, xid, yid, isw, y in dl_va:
                xseq, wd, tid, xid, yid, isw = (
                    xseq.to(device), wd.to(device), tid.to(device), xid.to(device),
                    yid.to(device), isw.to(device)
                )
                pred_log = model(xseq, wd, tid, xid, yid, isw).cpu().numpy()
                y_log = y.numpy()
                preds.append(np.expm1(pred_log))
                ys.append(np.expm1(y_log))

        yhat = np.concatenate(preds)
        ytrue = np.concatenate(ys)
        rmse = float(root_mean_squared_error(ytrue, yhat))
        print(f"epoch {ep}: val RMSE={rmse:.4f}")

        if rmse < best_rmse - 1e-4:
            best_rmse = rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print("Early stop.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def predict_lstm_embed(model, df_split, seq_len, batch_size=2048):
    device = next(model.parameters()).device
    ds = SeqDatasetEmbed(df_split, seq_len)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    model.eval()
    preds = []
    with torch.no_grad():
        for xseq, wd, tid, xid, yid, isw, _y in dl:
            xseq, wd, tid, xid, yid, isw = (
                xseq.to(device), wd.to(device), tid.to(device), xid.to(device),
                yid.to(device), isw.to(device)
            )
            pred_log = model(xseq, wd, tid, xid, yid, isw).cpu().numpy()
            preds.append(np.expm1(pred_log))
    return np.concatenate(preds)

In [ ]:
import torch, pickle

def save_lstm_pkl(model, path_pkl, config: dict):
    payload = {
        "state_dict": model.state_dict(),
        "config": config
    }
    torch.save(payload, path_pkl)   # 用 torch 的序列化最穩（副檔名可叫 .pkl）
    # 或你也可以用 pickle.dump(payload, open(...,"wb"))，但 torch.save 更常用

config = {
    "seq_len": SEQ_LEN,
    "hidden": 64,
    "layers": 1,
    "lr": 1e-3,

    "n_weekday": 7,
    "n_t": 48,
    "n_x": int(df["x"].max()) + 1,
    "n_y": int(df["y"].max()) + 1,

    "emb_wd": 2,
    "emb_t": 8,
    "emb_x": 16,
    "emb_y": 16,
}



In [ ]:
def run_all_models(train_df, val_df, test_df, seq_len=14):
    results = []

    # ---- baselines ----
    hist = fit_baseline_hist(train_df)

    for split_name, df_split in [("val", val_df), ("test", test_df)]:
        y = df_split["count"].to_numpy()

        yhat = predict_baseline_hist(df_split, hist)
        results.append(eval_metrics(f"baseline_hist ({split_name})", y, yhat))

        yhat = baseline_lagk(df_split, k=1)
        results.append(eval_metrics(f"baseline_lag1 ({split_name})", y, yhat))

        yhat = baseline_lagk(df_split, k=7)
        results.append(eval_metrics(f"baseline_lag7 ({split_name})", y, yhat))

    # ---- LGBM ----
    lgbm = train_lgbm(train_df, val_df)
    for split_name, df_split in [("val", val_df), ("test", test_df)]:
        y = df_split["count"].to_numpy()
        yhat = predict_lgbm(lgbm, df_split)
        results.append(eval_metrics(f"LGBM ({split_name})", y, yhat))

    # ---- LSTM ----
    lstm = train_lstm_embed(train_df, val_df, seq_len=seq_len)
    save_lstm_pkl(lstm, "lstm_sapporo.pkl", config) #存成PKL檔案
    for split_name, df_split in [("val", val_df), ("test", test_df)]:
        y = df_split["count"].to_numpy()
        yhat = predict_lstm_embed(lstm, df_split, seq_len=seq_len)
        results.append(eval_metrics(f"LSTM ({split_name})", y, yhat))

    return pd.DataFrame(results).sort_values(["model"]).reset_index(drop=True)

report = run_all_models(train_df, val_df, test_df, seq_len=SEQ_LEN)
report

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002245 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 950
[LightGBM] [Info] Number of data points in the train set: 165000, number of used features: 9
[LightGBM] [Info] Start training from score 11.369030
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[63]	valid_0's rmse: 3.99604	valid_0's l2: 15.9683
epoch 1: val RMSE=5.2878
epoch 2: val RMSE=4.5034
epoch 3: val RMSE=4.2956
epoch 4: val RMSE=4.1793
epoch 5: val RMSE=4.0819
epoch 6: val RMSE=4.0185
epoch 7: val RMSE=3.9560
epoch 8: val RMSE=3.9907
epoch 9: val RMSE=3.8969
epoch 10: val RMSE=3.9029
epoch 11: val RMSE=3.8574
epoch 12: val RMSE=3.9922
epoch 13: val RMSE=3.8936
epoch 14: val RMSE=3.9314
Early stop.


,model,MAE,RMSE,R2,SMAPE
0,LGBM (test),2.927526,4.240403,0.808889,0.329878
1,LGBM (val),2.898701,3.996037,0.820535,0.320485
2,LSTM (test),2.724712,3.926487,0.836138,0.310928
3,LSTM (val),2.752722,3.857366,0.832775,0.303682
4,baseline_hist (test),2.994966,4.111707,0.820314,0.332580
5,baseline_hist (val),2.938789,3.943090,0.825259,0.319787
6,baseline_lag1 (test),4.050629,6.224436,0.588215,0.446107
7,baseline_lag1 (val),4.041029,6.084467,0.583931,0.435175
8,baseline_lag7 (test),3.375171,4.615709,0.773563,0.392284
9,baseline_lag7 (val),3.685971,4.972637,0.722097,0.389603
